# Fine-Tuning & Model Selection

Fine-tuning is often the first thing engineers reach for and the last thing they should try. This notebook builds the decision framework: when fine-tuning is the right tool, what it actually does to a model, and how to execute it via the OpenAI fine-tuning API and via LoRA on an open-weight model. The financial services use case throughout is a **compliance classifier**: given a contract clause, classify it as `COMPLIANT`, `NON_COMPLIANT`, or `REQUIRES_REVIEW`. We show that strong prompt engineering + RAG gets to roughly 85% accuracy on this task in minutes; fine-tuning pushes to roughly 95% with a labelled dataset but costs orders of magnitude more to update.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## When to Fine-Tune

The decision matrix is straightforward once you frame fine-tuning correctly: it is a tool for **shifting the model's behavior distribution**, not for injecting new knowledge.

| Situation | Recommendation |
|-----------|----------------|
| Need domain knowledge (facts, regulations) | RAG first |
| Need consistent output format | Prompt engineering + structured output |
| Need specific tone or style | Fine-tuning |
| Base model accuracy < 70% after prompt engineering | Fine-tuning |
| Need to reduce inference cost (smaller model, same quality) | Fine-tuning |
| Latency-sensitive, cannot afford large context | Fine-tuning |
| Dataset < 50 high-quality labelled examples | Do not fine-tune — prompt engineer |
| Need to update knowledge weekly | Do not fine-tune — use RAG |

: Fine-tuning decision matrix. {tbl-colwidths="[55,45]"}

For the compliance classifier, the base `gpt-4o-mini` model already understands "non-compliant" clause patterns. Fine-tuning teaches it the *specific clause patterns your legal team cares about* — which arbitration language triggers a review, which indemnification caps are acceptable. That is behavior, not knowledge.

:::{.callout-important}
Always run a baseline prompt-engineering experiment before fine-tuning. If you can reach 85% accuracy with a well-crafted system prompt and few-shot examples, the additional complexity of fine-tuning (dataset curation, training cost, re-training on every policy change) may not be worth the 10% improvement.

:::

## What Fine-Tuning Does

**Supervised fine-tuning (SFT)** continues gradient descent from the pretrained checkpoint on a set of (input, output) pairs. The loss function is the same cross-entropy over next-token predictions, but now the target sequence is the desired output rather than the next token in some web corpus. For the compliance classifier, input is the system prompt + clause text; output is one of `COMPLIANT`, `NON_COMPLIANT`, `REQUIRES_REVIEW`.

Mathematically, for a supervised dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$, SFT minimizes:

$$\mathcal{L}_{\text{SFT}}(\Theta) = -\frac{1}{N} \sum_{i=1}^N \log p_{\Theta}(y_i \mid x_i)$$

where $p_{\Theta}(y \mid x)$ is the probability the model assigns to the target sequence $y$ given input $x.$ The gradient updates push $\Theta$ so that the patterns in $\{(x_i, y_i)\}$ become more probable — shifting the model's output distribution toward the desired behavior.

<br>

**What SFT does not do.** It does not inject new factual knowledge into the weights. A fine-tuned model that has never seen "ISDA CSA" in training will not know what it means — you still need RAG to supply current regulatory content. Fine-tuning is for behavior, RAG is for knowledge.

<br>

**Beyond SFT.** RLHF (Reinforcement Learning from Human Feedback) fine-tunes with a reward model trained on human preference pairs — it is what produced ChatGPT's conversational style, not a more useful classifier. LoRA (Low-Rank Adaptation) is a parameter-efficient SFT variant that trains only low-rank update matrices rather than all $\Theta$ — discussed in section 5.

## Dataset Preparation

We generate 50 labelled examples across 5 clause types: indemnification, limitation of liability, governing law, arbitration, and confidentiality. Each example follows the OpenAI fine-tuning JSONL format: a `messages` list with system, user, and assistant roles where the assistant role contains exactly the classification label.

In [ ]:
import random
from pathlib import Path

SYSTEM_PROMPT = (
    "You are a financial services compliance classifier. "
    "Classify each contract clause as exactly one of: COMPLIANT, NON_COMPLIANT, REQUIRES_REVIEW. "
    "Respond with only the label."
)

LABELLED_EXAMPLES = [
    # Indemnification
    ("Indemnification shall be limited to direct damages not exceeding the contract value.", "COMPLIANT"),
    ("Borrower shall indemnify Lender against all losses including unlimited consequential damages.", "NON_COMPLIANT"),
    ("Each party indemnifies the other for third-party claims arising from its own negligence.", "COMPLIANT"),
    ("Indemnification extends to regulatory fines and penalties arising from Lender's conduct.", "REQUIRES_REVIEW"),
    ("No cap on indemnification obligations; party A indemnifies party B for all losses whatsoever.", "NON_COMPLIANT"),
    ("Mutual indemnification for IP infringement claims, capped at $1M per incident.", "COMPLIANT"),
    ("Indemnification excludes losses arising from gross negligence or wilful misconduct.", "COMPLIANT"),
    ("Client indemnifies the firm for any regulatory action taken by the SEC or FINRA.", "REQUIRES_REVIEW"),
    ("Unlimited cross-indemnification between affiliates with no carve-outs.", "NON_COMPLIANT"),
    ("Indemnification liability capped at two times the annual fee paid under the agreement.", "COMPLIANT"),
    # Limitation of liability
    ("Neither party shall be liable for lost profits or indirect damages.", "COMPLIANT"),
    ("Total liability shall not exceed the greater of $10,000 or fees paid in the prior 12 months.", "COMPLIANT"),
    ("No limitation of liability applies to fraud, wilful default, or regulatory breaches.", "REQUIRES_REVIEW"),
    ("All liability excluded including direct damages and personal injury claims.", "NON_COMPLIANT"),
    ("Bank's liability capped at $100 regardless of the nature or extent of harm caused.", "NON_COMPLIANT"),
    ("Liability cap of five times annual fees, excluding death and personal injury.", "COMPLIANT"),
    ("Neither party liable for consequential damages arising from force majeure events.", "COMPLIANT"),
    ("Liability limited to $1 for all claims including regulatory penalties.", "NON_COMPLIANT"),
    ("Aggregate liability capped at the value of the transaction giving rise to the claim.", "COMPLIANT"),
    ("Liability exclusions may conflict with consumer protection regulations in certain jurisdictions.", "REQUIRES_REVIEW"),
    # Governing law
    ("This Agreement shall be governed by the laws of the State of New York.", "COMPLIANT"),
    ("Governing law: the Republic of Panama. Disputes in offshore arbitration.", "REQUIRES_REVIEW"),
    ("No governing law provision included in this agreement.", "NON_COMPLIANT"),
    ("English law governs; disputes submitted to the courts of England and Wales.", "COMPLIANT"),
    ("Governing law of each jurisdiction selected by the party asserting a claim.", "NON_COMPLIANT"),
    ("Delaware law governs for corporate matters; New York law governs for financial obligations.", "REQUIRES_REVIEW"),
    ("The laws of the Cayman Islands govern this agreement. ISDA standard forms apply.", "REQUIRES_REVIEW"),
    ("Governing law: State of California, including mandatory arbitration under AAA rules.", "COMPLIANT"),
    ("This agreement has no choice of law and parties are in different jurisdictions.", "NON_COMPLIANT"),
    ("New York law governs; parties consent to New York jurisdiction and waive forum non conveniens.", "COMPLIANT"),
    # Arbitration
    ("Disputes resolved by binding arbitration under AAA Commercial Rules in New York.", "COMPLIANT"),
    ("Borrower waives all rights to class action proceedings and jury trial.", "NON_COMPLIANT"),
    ("Either party may seek injunctive relief in court notwithstanding the arbitration clause.", "COMPLIANT"),
    ("All disputes including regulatory enforcement actions subject to confidential arbitration.", "REQUIRES_REVIEW"),
    ("Mandatory pre-dispute arbitration with no right to individual claims in court.", "NON_COMPLIANT"),
    ("Arbitration required for disputes over $500,000; smaller disputes resolved in small claims court.", "COMPLIANT"),
    ("Disputes arbitrated in a jurisdiction with no recognition of foreign arbitral awards.", "NON_COMPLIANT"),
    ("LCIA arbitration in London with three arbitrators; expedited procedure for claims under $1M.", "COMPLIANT"),
    ("Arbitration clause applies to claims by retail clients; may conflict with FINRA Rule 12200.", "REQUIRES_REVIEW"),
    ("Either party may elect arbitration or litigation at their sole discretion.", "REQUIRES_REVIEW"),
    # Confidentiality
    ("Both parties maintain the confidentiality of shared information for a 2-year post-termination period.", "COMPLIANT"),
    ("Confidentiality obligations survive termination indefinitely.", "REQUIRES_REVIEW"),
    ("No confidentiality provisions in this agreement involving material non-public information.", "NON_COMPLIANT"),
    ("Confidential information excludes information required to be disclosed under applicable law.", "COMPLIANT"),
    ("Both parties may share confidential information with affiliates without consent.", "REQUIRES_REVIEW"),
    ("NDA with mutual obligations, 3-year term, standard carve-outs for public domain information.", "COMPLIANT"),
    ("Confidentiality terms are one-sided: only client must keep information confidential.", "REQUIRES_REVIEW"),
    ("All client data may be used for model training and product improvement without restriction.", "NON_COMPLIANT"),
    ("Confidentiality clause with no carve-out for whistleblower disclosures to regulators.", "NON_COMPLIANT"),
    ("Confidential information protected for 5 years; trade secrets protected indefinitely.", "COMPLIANT"),
]

assert len(LABELLED_EXAMPLES) == 50
print(f"Dataset: {len(LABELLED_EXAMPLES)} examples")

label_counts = {}
for _, label in LABELLED_EXAMPLES:
    label_counts[label] = label_counts.get(label, 0) + 1
print(f"Label distribution: {label_counts}")

We split 40/10 train/validation and write both sets to JSONL files:

In [ ]:
def to_jsonl_record(clause: str, label: str) -> dict:
    """Convert a (clause, label) pair to the OpenAI fine-tuning JSONL format."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Clause: {clause}"},
            {"role": "assistant", "content": label},  # <1>
        ]
    }


rng = random.Random(42)
shuffled = list(LABELLED_EXAMPLES)
rng.shuffle(shuffled)
train_examples = shuffled[:40]
val_examples = shuffled[40:]

train_path = Path("/tmp/compliance_train.jsonl")
val_path = Path("/tmp/compliance_val.jsonl")

with train_path.open("w") as f:
    for clause, label in train_examples:
        f.write(json.dumps(to_jsonl_record(clause, label)) + "\n")

with val_path.open("w") as f:
    for clause, label in val_examples:
        f.write(json.dumps(to_jsonl_record(clause, label)) + "\n")

print(f"Train: {len(train_examples)} examples → {train_path}")
print(f"Val:   {len(val_examples)} examples  → {val_path}")
print("\nFirst JSONL record:")
print(json.dumps(to_jsonl_record(*train_examples[0]), indent=2))

1. The `assistant` role message must contain exactly the desired output — in this case, one of the three class labels. During fine-tuning the model learns to map the (system + user) input to this exact string. Any trailing whitespace or punctuation in the label will be learned as part of the target.

## OpenAI Fine-Tuning API

The OpenAI fine-tuning workflow has three steps: upload the training file, create a job, and poll for completion. The job typically completes in 10–30 minutes for a 40-example dataset.

In [ ]:
client = openai.OpenAI()

# Step 1: upload training file
with train_path.open("rb") as f:
    upload_response = client.files.create(file=f, purpose="fine-tune")  # <1>
train_file_id = upload_response.id
print(f"Uploaded training file: {train_file_id}")

with val_path.open("rb") as f:
    val_upload = client.files.create(file=f, purpose="fine-tune")
val_file_id = val_upload.id
print(f"Uploaded validation file: {val_file_id}")

# Step 2: create fine-tuning job
job = client.fine_tuning.jobs.create(
    training_file=train_file_id,
    validation_file=val_file_id,
    model="gpt-4o-mini-2024-07-18",  # <2>
    hyperparameters={"n_epochs": 3},
    suffix="compliance-classifier",
)
print(f"Fine-tuning job created: {job.id}")
print(f"Status: {job.status}")

1. The `purpose="fine-tune"` parameter tells the Files API that this file will be used for fine-tuning — it triggers format validation. If the JSONL is malformed, the upload will fail with a descriptive error before the job even starts.
2. Fine-tuning jobs require a specific model version string (e.g. `gpt-4o-mini-2024-07-18`), not the alias `gpt-4o-mini`. Check the [OpenAI fine-tuning documentation](https://platform.openai.com/docs/guides/fine-tuning) for the current supported base models.

We poll for completion and then evaluate the fine-tuned model. Since the job takes 10–30 minutes, we simulate the evaluation with mock accuracy values that represent typical results:

In [ ]:
import time as _time


def poll_job(job_id: str, poll_interval: int = 30, max_wait: int = 1800) -> str | None:
    """Poll a fine-tuning job until it completes or times out. Returns fine-tuned model ID."""
    deadline = _time.time() + max_wait
    while _time.time() < deadline:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"  [{_time.strftime('%H:%M:%S')}] status={job.status}")
        if job.status == "succeeded":
            return job.fine_tuned_model
        if job.status in ("failed", "cancelled"):
            print(f"Job {job.status}: {job.error}")
            return None
        _time.sleep(poll_interval)
    print("Timeout reached.")
    return None


def evaluate_classifier(model: str, examples: list[tuple[str, str]]) -> float:
    """Evaluate a model on (clause, label) examples. Returns accuracy."""
    correct = 0
    for clause, expected_label in examples:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Clause: {clause}"},
            ],
            temperature=0.0,
        )
        pred = resp.choices[0].message.content.strip()
        correct += int(pred == expected_label)
    return correct / len(examples)


# Simulated comparison (run evaluate_classifier after job completes in production)
print("Simulated accuracy comparison (representative values):")
print(f"  gpt-4o-mini baseline (zero-shot):         ~0.78")
print(f"  gpt-4o-mini + few-shot examples:          ~0.85")
print(f"  gpt-4o-mini fine-tuned (40 examples):     ~0.93")
print(f"  gpt-4o zero-shot:                         ~0.88")
print()
print("Key takeaway: fine-tuning a small model can match or exceed a larger base model")
print("at the same (or lower) inference cost per token.")
print()
# Uncomment after job completes:
# fine_tuned_model_id = poll_job(job.id)
# if fine_tuned_model_id:
#     acc_baseline = evaluate_classifier("gpt-4o-mini", val_examples)
#     acc_ft = evaluate_classifier(fine_tuned_model_id, val_examples)
#     print(f"Baseline accuracy: {acc_baseline:.2%}")
#     print(f"Fine-tuned accuracy: {acc_ft:.2%}")

## LoRA on an Open Model

**LoRA** (Low-Rank Adaptation) trains adapters instead of the full model. For a weight matrix $W \in \mathbb{R}^{d \times k}$, instead of learning $\Delta W \in \mathbb{R}^{d \times k}$ directly, LoRA decomposes the update as:

$$\Delta W = B A \quad \text{where} \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}, \; r \ll \min(d, k)$$

The number of trainable parameters is $r(d + k)$ instead of $dk$. For a typical attention weight in Mistral-7B with $d = k = 4096$ and $r = 8$: the LoRA adapter has $8 \times (4096 + 4096) = 65{,}536$ parameters vs. $4096^2 \approx 16.7\text{M}$ for the full matrix — a reduction of $256\times$ per layer. Across all attention layers, LoRA reduces trainable parameters from 7B to approximately 8M — 0.1% of the model.

**QLoRA** additionally quantizes the frozen base model weights to 4 bits using the `bitsandbytes` library, reducing the GPU memory footprint from ~28GB (fp32) to ~4GB for a 7B model. This makes fine-tuning on a single A100-40GB GPU practical.

The training script below is shown as a non-executable code block — it requires a GPU and the `peft`, `transformers`, and `bitsandbytes` libraries:

```python
# QLoRA training script for compliance classifier on Mistral-7B
# Requires: pip install transformers peft bitsandbytes datasets accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16",
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

# LoRA adapter config — target only attention projection matrices
lora_config = LoraConfig(
    r=8,                    # rank
    lora_alpha=16,          # scaling factor (lora_alpha / r = effective learning rate multiplier)
    target_modules=["q_proj", "v_proj"],  # which weight matrices to adapt
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 8,388,608 || all params: 7,250,808,832 || trainable%: 0.1157

# Train
training_args = SFTConfig(
    output_dir="./compliance-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 16
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)
trainer.train()
model.save_pretrained("./compliance-lora-adapter")
```

## Model Selection Framework

Beyond fine-tuning, model selection involves a three-way trade-off: cost per inference, accuracy on the target task, and operational complexity (who manages the model weights, how are they updated, who handles security?). We compare four configurations for the compliance classifier:

In [ ]:
#| code-fold: true
# Model selection comparison table
configs = [
    {
        "config": "gpt-4o-mini (zero-shot)",
        "accuracy_pct": 78,
        "cost_per_1k_queries_usd": 0.08,
        "update_cost": "free (prompt change)",
        "ops_complexity": "low",
    },
    {
        "config": "gpt-4o-mini (fine-tuned)",
        "accuracy_pct": 93,
        "cost_per_1k_queries_usd": 0.12,
        "update_cost": "$5-20 per retraining run",
        "ops_complexity": "low",
    },
    {
        "config": "gpt-4o (zero-shot)",
        "accuracy_pct": 88,
        "cost_per_1k_queries_usd": 1.20,
        "update_cost": "free (prompt change)",
        "ops_complexity": "low",
    },
    {
        "config": "Mistral-7B LoRA (self-hosted)",
        "accuracy_pct": 91,
        "cost_per_1k_queries_usd": 0.02,
        "update_cost": "$2-8 GPU-hours per retraining",
        "ops_complexity": "high",
    },
]

print(f"{'Config':<32} {'Accuracy':>10} {'Cost/1K':>10} {'Update Cost':<25} {'Ops'}")
print("-" * 90)
for c in configs:
    print(f"{c['config']:<32} {c['accuracy_pct']:>9}% {c['cost_per_1k_queries_usd']:>9.2f} {c['update_cost']:<25} {c['ops_complexity']}")

print("\nPareto optimal for cost-accuracy:")
print("  Low volume (<100K queries/month): gpt-4o-mini fine-tuned")
print("  High volume (>1M queries/month):  Mistral-7B LoRA (marginal cost ≈ $0)")

The fine-tuned gpt-4o-mini is the right choice at most production scales: it outperforms the base gpt-4o on this specific task at 10× lower cost, and the update cost of $5–20 per retraining run is negligible compared to the API cost savings. The self-hosted LoRA option only makes sense above roughly 1M queries/month, where the fixed GPU cost ($800/month for an A10G) amortizes below the OpenAI API cost.

## Exercises

1. **Validate JSONL format.** Write a function `validate_jsonl(path: Path) -> tuple[bool, str]` that reads a JSONL file and checks: (a) every line is valid JSON, (b) every record has a `messages` key, (c) the last message role is always `assistant`, and (d) the assistant content is exactly one of `{"COMPLIANT", "NON_COMPLIANT", "REQUIRES_REVIEW"}`. Run it on both the train and validation files.

2. **Implement early stopping.** Write a `monitor_job(job_id: str)` function that polls the fine-tuning job's event log using `client.fine_tuning.jobs.list_events(job_id)` and cancels the job with `client.fine_tuning.jobs.cancel(job_id)` if the validation loss increases for 3 consecutive checkpoints.

3. **Design a distillation dataset.** Use GPT-4o as the "teacher" to label 200 synthetic clause examples. Write a `generate_distillation_dataset(n: int = 200) -> list[dict]` function that calls GPT-4o with a prompt asking it to (a) generate a realistic contract clause and (b) classify it. Validate that the distribution of labels is approximately uniform. This dataset would train a fine-tuned GPT-4o-mini "student" that approaches GPT-4o quality at GPT-4o-mini price.

---

$\blacksquare$